# Customer Segmentation - Amazing International Airlines (AIAI)

**Group 68** | Data Mining Assignment - Delivery 2 | December 2025

This notebook implements and evaluates clustering algorithms to identify distinct customer segments within AIAI's loyalty program based on the exploratory analysis performed in Delivery 1.

## 1. Data Preparation & Feature Selection

Loading the cleaned dataset from Delivery 1 and selecting features for clustering based on segmentation relevance.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Create directory for plots
output_dir = 'Plots'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Load the cleaned dataset from Delivery 1
df = pd.read_csv('Data/DM_AIAI_FinalCustomerSegments.csv')
df = df.sample(n=50000, random_state=42)
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Unique customers: {df['Loyalty#'].nunique():,}")

In [ ]:
# Select features for clustering based on Delivery 1 analysis

clustering_features = [
    'Customer Lifetime Value',      # Primary value indicator
    'NumFlights',                    # Activity level
    'DistanceKM',                    # Travel intensity
    'PointsAccumulated',             # Loyalty engagement
    'avg_km_per_flight',             # Travel pattern (short vs long haul)
    'avg_points_per_flight',         # Reward efficiency
    'companion_ratio',               # Social travel behavior
    'clv_per_flight',                # Value efficiency
    'months_since_flight',           # Recency
    'active_days'                    # Tenure
]

# Check if all features exist
available_features = [f for f in clustering_features if f in df.columns]
missing_features = [f for f in clustering_features if f not in df.columns]

# Create clustering dataset
df_clustering = df[available_features].copy()

print(f"Shape: {df_clustering.shape}")
for i, feat in enumerate(available_features, 1):
    print(f"  {i}. {feat}")

## 2. Data Scaling & Normalization

Applying robust scaling to handle outliers identified in Delivery 1 EDA.

In [ ]:
# Apply RobustScaler (better for data with outliers)
scaler = RobustScaler()
df_scaled = scaler.fit_transform(df_clustering)

# Convert back to DataFrame for easier handling
df_scaled = pd.DataFrame(df_scaled, columns=df_clustering.columns, index=df_clustering.index)

display(df_scaled.describe().T)

## 3. K-Means Clustering Implementation

Applying K-Means clustering with 4 clusters, based on the findings from delivery 1.

In [ ]:
optimal_k = 4

# Apply K-Means clustering
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10, max_iter=300)
df['KMeans_Cluster'] = kmeans.fit_predict(df_scaled)

print(f"=== K-MEANS CLUSTERING (k={optimal_k}) ===")
print(f"\nCluster Distribution:")
print(df['KMeans_Cluster'].value_counts().sort_index())

# Calculate clustering metrics
sil_score = silhouette_score(df_scaled, df['KMeans_Cluster'])
db_score = davies_bouldin_score(df_scaled, df['KMeans_Cluster'])
ch_score = calinski_harabasz_score(df_scaled, df['KMeans_Cluster'])

print(f"\n=== CLUSTERING QUALITY METRICS ===")
print(f"Silhouette Score: {sil_score:.3f} (higher is better, range: -1 to 1)")
print(f"Davies-Bouldin Index: {db_score:.3f} (lower is better)")
print(f"Calinski-Harabasz Score: {ch_score:.2f} (higher is better)")

In [ ]:
# Plot K-Means Clusters using PCA for 2D visualization

pca = PCA(n_components=2, random_state=42)
pca_components = pca.fit_transform(df_scaled)
df_pca = pd.DataFrame(data=pca_components, columns=['PCA1', 'PCA2'], index=df.index)
df_pca['KMeans_Cluster'] = df['KMeans_Cluster']
plt.figure(figsize=(10, 7))
sns.scatterplot(data=df_pca, x='PCA1', y='PCA2', hue='KMeans_Cluster', palette='Set2', s=50, alpha=0.7)
plt.title('K-Means Clusters Visualized with PCA', fontsize=14, fontweight='bold')
plt.xlabel('PCA Component 1', fontsize=12)
plt.ylabel('PCA Component 2', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=12)
plt.tight_layout()
# plt.savefig(f'{output_dir}/kmeans_clusters_pca.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Agglomerative Hierarchical Clustering Implementation

Applying Hierarchical Clustering with Ward linkage for comparison with K-Means.

In [ ]:
# Apply Agglomerative Hierarchical Clustering with Ward linkage
hierarchical = AgglomerativeClustering(n_clusters=optimal_k, linkage='ward')
df['Hierarchical_Cluster'] = hierarchical.fit_predict(df_scaled)

print(f"=== AGGLOMERATIVE HIERARCHICAL CLUSTERING (k={optimal_k}, Ward Linkage) ===")
print(f"\nCluster Distribution:")
print(df['Hierarchical_Cluster'].value_counts().sort_index())

# Calculate clustering metrics for Hierarchical
sil_score_hier = silhouette_score(df_scaled, df['Hierarchical_Cluster'])
db_score_hier = davies_bouldin_score(df_scaled, df['Hierarchical_Cluster'])
ch_score_hier = calinski_harabasz_score(df_scaled, df['Hierarchical_Cluster'])

print(f"\n=== CLUSTERING QUALITY METRICS ===")
print(f"Silhouette Score: {sil_score_hier:.3f} (higher is better, range: -1 to 1)")
print(f"Davies-Bouldin Index: {db_score_hier:.3f} (lower is better)")
print(f"Calinski-Harabasz Score: {ch_score_hier:.2f} (higher is better)")

In [ ]:
# Plot Hierarchical Clusters using PCA for 2D visualization
df_pca['Hierarchical_Cluster'] = df['Hierarchical_Cluster']

plt.figure(figsize=(10, 7))
sns.scatterplot(data=df_pca, x='PCA1', y='PCA2', hue='Hierarchical_Cluster', palette='Set1', s=50, alpha=0.7)
plt.title('Hierarchical Clusters Visualized with PCA (Ward Linkage)', fontsize=14, fontweight='bold')
plt.xlabel('PCA Component 1', fontsize=12)
plt.ylabel('PCA Component 2', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=12)
plt.tight_layout()
plt.savefig(f'{output_dir}/hierarchical_clusters_pca.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Comparison of Clustering Algorithms

Comparing K-Means and Hierarchical Clustering performance.

In [ ]:
# Compare clustering metrics
comparison_df = pd.DataFrame({
    'Algorithm': ['K-Means', 'Hierarchical (Ward)'],
    'Silhouette Score': [sil_score, sil_score_hier],
    'Davies-Bouldin Index': [db_score, db_score_hier],
    'Calinski-Harabasz Score': [ch_score, ch_score_hier]
})

print("=== ALGORITHM COMPARISON ===")
display(comparison_df)

# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = ['Silhouette Score', 'Davies-Bouldin Index', 'Calinski-Harabasz Score']
colors = ['#2ecc71', '#e74c3c', '#3498db']

for idx, (metric, color) in enumerate(zip(metrics, colors)):
    axes[idx].bar(comparison_df['Algorithm'], comparison_df[metric], color=color, alpha=0.7, edgecolor='black')
    axes[idx].set_ylabel(metric, fontsize=11)
    axes[idx].set_title(metric, fontsize=12, fontweight='bold')
    axes[idx].grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, v in enumerate(comparison_df[metric]):
        axes[idx].text(i, v, f'{v:.2f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{output_dir}/clustering_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# Select best algorithm based on metrics
best_silhouette = comparison_df.loc[comparison_df['Silhouette Score'].idxmax(), 'Algorithm']
best_db = comparison_df.loc[comparison_df['Davies-Bouldin Index'].idxmin(), 'Algorithm']
best_ch = comparison_df.loc[comparison_df['Calinski-Harabasz Score'].idxmax(), 'Algorithm']

print("\n=== BEST ALGORITHM BY METRIC ===")
print(f"Highest Silhouette Score: {best_silhouette}")
print(f"Lowest Davies-Bouldin Index: {best_db}")
print(f"Highest Calinski-Harabasz Score: {best_ch}")

**Algorithm Selection:**

Based on the comparison metrics above, select the algorithm that performs best overall for the final segmentation. Consider:
- **Silhouette Score**: Measures how similar objects are to their own cluster vs other clusters (higher is better)
- **Davies-Bouldin Index**: Average similarity ratio of each cluster with its most similar cluster (lower is better)
- **Calinski-Harabasz Score**: Ratio of between-cluster to within-cluster dispersion (higher is better)

For the remainder of the analysis, we'll use the **best-performing algorithm** based on these metrics.

## 6. Cluster Profiling & Interpretation

Analyzing cluster characteristics to create meaningful business segments.

In [ ]:
# Select which clustering algorithm to use for final segmentation
# Change this to 'Hierarchical_Cluster' if hierarchical performs better
selected_algorithm = 'KMeans_Cluster'  # or 'Hierarchical_Cluster'

# Create cluster profiles using original (non-scaled) features
cluster_profiles = df.groupby(selected_algorithm)[available_features].mean()

print(f"=== CLUSTER PROFILES (Mean Values) - Using {selected_algorithm} ===")
display(cluster_profiles.T)

# Add demographic information if available
if 'LoyaltyStatus' in df.columns:
    print("\n=== LOYALTY STATUS DISTRIBUTION BY CLUSTER ===")
    loyalty_dist = pd.crosstab(df[selected_algorithm], df['LoyaltyStatus'], normalize='index') * 100
    display(loyalty_dist.round(1))

In [ ]:
# Visualize cluster profiles with radar chart
from math import pi

fig = plt.figure(figsize=(14, 10))

# Normalize cluster profiles for radar chart (0-1 scale)
cluster_profiles_norm = (cluster_profiles - cluster_profiles.min()) / (cluster_profiles.max() - cluster_profiles.min())

# Create subplot for each cluster
for idx, cluster in enumerate(cluster_profiles_norm.index):
    ax = plt.subplot(2, 2, idx + 1, projection='polar')
    
    # Prepare data
    categories = list(cluster_profiles_norm.columns)
    values = cluster_profiles_norm.loc[cluster].values.tolist()
    values += values[:1]  # Complete the circle
    
    # Angles for each variable
    angles = [n / len(categories) * 2 * pi for n in range(len(categories))]
    angles += angles[:1]
    
    # Plot
    ax.plot(angles, values, 'o-', linewidth=2, label=f'Cluster {cluster}')
    ax.fill(angles, values, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=8)
    ax.set_ylim(0, 1)
    ax.set_title(f'Cluster {cluster}', size=12, fontweight='bold', pad=20)
    ax.grid(True)

plt.tight_layout()
plt.savefig(f'{output_dir}/cluster_profiles_radar.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Cluster profiles visualized")

## 7. Business Interpretation & Segment Naming

Assigning meaningful names to clusters based on their characteristics.

In [ ]:
# Assign business names to clusters based on their characteristics
# ADJUST THESE MAPPINGS BASED ON YOUR ACTUAL CLUSTER PROFILES

cluster_names = {
    0: 'VIP Frequent Flyers',
    1: 'Occasional Travelers', 
    2: 'Regular Customers',
    3: 'At-Risk/Dormant'
}

df['Segment_Name'] = df[selected_algorithm].map(cluster_names)
df['Final_Cluster'] = df[selected_algorithm]

print("=== CUSTOMER SEGMENT DEFINITIONS ===\n")
for cluster_id, name in cluster_names.items():
    count = (df[selected_algorithm] == cluster_id).sum()
    percentage = (count / len(df)) * 100
    print(f"Cluster {cluster_id}: {name}")
    print(f"  Size: {count:,} customers ({percentage:.1f}%)")
    print(f"  Characteristics:")
    
    # Display key metrics
    cluster_data = df[df[selected_algorithm] == cluster_id]
    print(f"    - Avg CLV: ${cluster_data['Customer Lifetime Value'].mean():,.2f}")
    print(f"    - Avg Flights: {cluster_data['NumFlights'].mean():.1f}")
    print(f"    - Avg Distance: {cluster_data['DistanceKM'].mean():,.0f} km")
    if 'months_since_flight' in cluster_data.columns:
        print(f"    - Avg Recency: {cluster_data['months_since_flight'].mean():.1f} months")
    print()

## 8. Export Final Segmented Dataset

Saving the clustered data for marketing strategy development.

In [ ]:
# Export final segmented dataset
df.to_csv('Data/DM_AIAI_FinalCustomerSegments.csv', index=False)

print("=== EXPORT SUMMARY ===")
print(f"✓ Exported {len(df):,} customer records with cluster assignments")
print(f"✓ Saved to: Data/DM_AIAI_FinalCustomerSegments.csv")

print("\n=== FINAL SEGMENT DISTRIBUTION ===")
segment_summary = df['Segment_Name'].value_counts()
for segment, count in segment_summary.items():
    print(f"{segment}: {count:,} customers ({count/len(df)*100:.1f}%)")

---

## Summary

**Clustering Results:**
- Algorithm: K-Means
- Optimal Clusters: 4 (adjust based on your analysis)
- Silhouette Score: [Will be displayed after running]
- Total Customers Segmented: [Will be displayed after running]

**Key Segments Identified:**
1. **VIP Frequent Flyers**: High CLV, frequent travel, premium service targets
2. **Regular Customers**: Moderate activity, loyalty program core
3. **Occasional Travelers**: Low frequency, growth potential
4. **At-Risk/Dormant**: Low recent activity, retention focus

**Next Steps:**
- Develop targeted marketing strategies for each segment
- Create personalized communications and offers
- Monitor segment migration over time
- Validate segments with business stakeholders